identify the env

identify the node (env/node/rssi)

two dataset strategy:

a. data from all env -> into a dataset and split to train 0.75/test 0.25

b. data from 3 env -> for train / 1 env -> test


100/500/1000 -> seq length as frame

model: CNN

overlapping: 40%/ 50%

report:

model.summary() 

hyper parameter (overlapping / learning rate)

In [34]:
# 📓 Jupyter Notebook: Preprocess e1-n0 dataset

import pandas as pd
import numpy as np

# === Step 1: Load CSV ===
df = pd.read_csv("/home/hzyc/iot-ml/resnet/env/e4-garden.csv")  # 改成你的檔案路徑

# === Step 2: 基本清理 ===
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts')

# === Step 3: 設定環境 ===
df['env'] = 1

# === Step 4: 篩選 e1-n0 ===
df_n0 = df[df['device'] == 'RIOT-BLE-0'].copy()

# === Step 5: 差分 ===
df_n0['rssi_diff'] = df_n0['rssi'].diff()

# === Step 6: normalization (min-max) ===
y_min = df_n0['rssi_diff'].min()
y_max = df_n0['rssi_diff'].max()
df_n0['rssi_norm'] = (df_n0['rssi_diff'] - y_min) / (y_max - y_min)

# === Step 7: 建立 timeslot（每100個封包一段）===
# 先確保順序正確
df_n0 = df_n0.sort_values('ts').reset_index(drop=True)
# 每100筆為一個 timeslot
WINDOW_SIZE = 100
df_n0['timeslot'] = df_n0.index // WINDOW_SIZE
# === Step 8: 統一欄位名稱 ===
df_n0['node'] = 'n0'
df_n0_final = df_n0[['timeslot', 'env', 'node', 'rssi_norm']].rename(columns={'rssi_norm': 'rssi'})

# === Step 9: 顯示結果 ===
df_n0_final.head(101)

,timeslot,env,node,rssi
0,0,1,n0,NaN
1,0,1,n0,0.666667
2,0,1,n0,0.481481
3,0,1,n0,0.592593
4,0,1,n0,0.370370
...,...,...,...,...
96,0,1,n0,0.518519
97,0,1,n0,0.518519
98,0,1,n0,0.370370
99,0,1,n0,0.481481


In [35]:
import pandas as pd
import numpy as np
import glob

# === 參數設定 ===
WINDOW_SIZE = 500 # 100/500/1000
STRIDE = 40 # 40/50

# === 檔案路徑 ===
env_files = [
    "env/e0-bridge.csv",
    "env/e1-lake.csv",
    "env/e2-forest.csv",
    "env/e3-river.csv",
    "env/e4-garden.csv"
]

# === device mapping ===
device_to_label = {
    "RIOT-BLE-0": 0,
    "RIOT-BLE-1": 1,
    "RIOT-BLE-2": 2,
    "RIOT-BLE-3": 3
}

node_files = [
    "node/node0.csv",
    "node/node1.csv",
    "node/node2.csv",
    "node/node3.csv"
]

env_to_label = {
    "env/e0-bridge.csv": 0,
    "env/e1-lake.csv": 1,
    "env/e2-forest.csv": 2,
    "env/e3-river.csv": 3,
    "env/e4-garden.csv": 4
}

In [36]:
# === create node csvs ===
from pathlib import Path

all_df = []

for file in env_files:
    df = pd.read_csv(file)  

    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df = df.dropna(subset=["ts"]).copy()
    df["source_file"] = file

    all_df.append(df)

df_all = pd.concat(all_df, ignore_index=True)

Path("node").mkdir(exist_ok=True)
for device, node_name in device_to_label.items():
    sub = df_all[df_all["device"] == device].copy()
    sub = sub.sort_values("ts")
    sub.to_csv(f"node/node{node_name}.csv", index=False)
    
    print(f"node{node_name}.csv:  {len(sub)} rows")

node0.csv:  98602 rows
node1.csv:  82109 rows
node2.csv:  77771 rows
node3.csv:  82140 rows


In [38]:
# === 儲存結果 ===
X = []
y = []

for node_id, file in enumerate(node_files):
    df = pd.read_csv(file)

    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df = df.dropna(subset=["ts"]).sort_values("ts")

    for env_name, env_label in env_to_label.items():
        df_env = df[df["source_file"] == env_name].copy()

        if len(df_env) < WINDOW_SIZE:
            continue

        df_env["rssi_diff"] = df_env["rssi"].diff()

        df_env = df_env.dropna(subset=["rssi_diff"])

        y_min = df_env["rssi_diff"].min()
        y_max = df_env["rssi_diff"].max()

        if y_max - y_min == 0:
            continue

        df_env["rssi_norm"] = (df_env["rssi_diff"] - y_min) / (y_max - y_min)

        data = df_env["rssi_norm"].values

        for i in range(0, len(data) - WINDOW_SIZE, STRIDE):
            seq = data[i:i+WINDOW_SIZE]
            X.append(seq)
            y.append(env_label)

X = np.array(X)
y = np.array(y)
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8277, 500)
y shape: (8277,)


In [39]:
# =========================
# Step 1: reshape for 1D CNN
# =========================
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

X = X.astype(np.float32)
y = y.astype(np.int64)

# PyTorch Conv1d input: (batch, channels, length)
X_data = X[:, np.newaxis, :]   # -> (6779, 1, 100)

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# turn into PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

# create Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("X_train:", X_train_tensor.shape)
print("X_test :", X_test_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("y_test :", y_test_tensor.shape)

X_train: torch.Size([6207, 1, 500])
X_test : torch.Size([2070, 1, 500])
y_train: torch.Size([6207])
y_test : torch.Size([2070])


In [40]:
train_files_3env = [
    "e0-bridge.csv",
    "e1-lake.csv",
    "e2-forest.csv"
]

test_files_1env = [
    "e3-river.csv"
]

In [41]:
# === Residual Block ===
import torch.nn as nn

class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels) # batch norm after conv1
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv1d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels) # batch norm after conv2

        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels: # projection shortcut if dimensions differ
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
    
    # forward pass
    def forward(self, x):
        identity = self.shortcut(x)

        # conv1 -> bn -> relu
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        # conv2 -> bn
        out = self.conv2(out)
        out = self.bn2(out)

        # add shortcut
        out += identity
        out = self.relu(out)

        return out

In [42]:
# === ResNet1D Model ===

class ResNet1D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # initial convolution and pooling
        self.stem = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )

        # layer1
        self.layer1 = nn.Sequential(
            ResidualBlock1D(16, 16, stride=1)
        )

        # layer2
        self.layer2 = nn.Sequential(
            ResidualBlock1D(16, 32, stride=2)
        )

        # layer3
        self.layer3 = nn.Sequential(
            ResidualBlock1D(32, 64, stride=2)
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.stem(x)         # (B, 1, 100) -> (B, 16, 25)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x)  # (B, 64, 1)
        x = x.squeeze(-1)        # (B, 64, 1) -> (B, 64)
        x = self.fc(x)           # (B, num_classes)
        return x

num_classes = len(np.unique(y))
model = ResNet1D(num_classes=num_classes)
print(model)

ResNet1D(
  (stem): Sequential(
    (0): Conv1d(1, 16, kernel_size=(7,), stride=(2,), padding=(3,), bias=False)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
  )
  (layer2): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,), bias=False)
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=T

In [43]:
# =========================
# Step 3: training setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

In [44]:
# =========================
# Step 4: training loop
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            loss = criterion(outputs, yb)

            total_loss += loss.item() * xb.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return total_loss / total, correct / total

In [45]:
# =========================
# Step 5: run training
# =========================
num_epochs = 20
# best_test_loss = float("inf")
# best_state = None

for epoch in range(num_epochs):
      train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
      test_loss, test_acc = evaluate(model, test_loader, criterion, device)

      print(f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

# save best model
# if test_loss < best_test_loss:
#     best_test_loss = test_loss
#     best_state = model.state_dict()

Epoch [1/20] Train Loss: 1.0757, Train Acc: 0.5679 | Test Loss: 0.8563, Test Acc: 0.6430
Epoch [2/20] Train Loss: 0.7506, Train Acc: 0.6860 | Test Loss: 0.6339, Test Acc: 0.7449
Epoch [3/20] Train Loss: 0.5636, Train Acc: 0.7825 | Test Loss: 0.4513, Test Acc: 0.8420
Epoch [4/20] Train Loss: 0.4814, Train Acc: 0.8196 | Test Loss: 0.3799, Test Acc: 0.8841
Epoch [5/20] Train Loss: 0.3891, Train Acc: 0.8658 | Test Loss: 0.4075, Test Acc: 0.8589
Epoch [6/20] Train Loss: 0.3283, Train Acc: 0.8898 | Test Loss: 0.2693, Test Acc: 0.9014
Epoch [7/20] Train Loss: 0.3085, Train Acc: 0.8950 | Test Loss: 0.2434, Test Acc: 0.9285
Epoch [8/20] Train Loss: 0.2910, Train Acc: 0.9004 | Test Loss: 0.2194, Test Acc: 0.9232
Epoch [9/20] Train Loss: 0.2498, Train Acc: 0.9183 | Test Loss: 0.2544, Test Acc: 0.8923
Epoch [10/20] Train Loss: 0.2247, Train Acc: 0.9275 | Test Loss: 0.1221, Test Acc: 0.9729
Epoch [11/20] Train Loss: 0.1966, Train Acc: 0.9397 | Test Loss: 0.1415, Test Acc: 0.9536
Epoch [12/20] Train

In [46]:
# =========================
# Step 6: prediction example
# =========================

# model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    sample_x = X_test_tensor[:5].to(device)
    outputs = model(sample_x)
    preds = outputs.argmax(dim=1).cpu().numpy()

print("Pred:", preds)
print("True:", y_test[:5])

Pred: [2 3 1 0 3]
True: [2 3 1 0 3]


In [47]:
# =========================
# Step 7: confusion matrix
# =========================
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_true.extend(yb.numpy())

cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(all_true, all_preds))

Confusion Matrix:
 [[486   0   1   0   0]
 [  0 266   3   0   1]
 [  0   0 463   1   6]
 [  0   0   9 447   3]
 [  0   0   0   0 384]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       487
           1       1.00      0.99      0.99       270
           2       0.97      0.99      0.98       470
           3       1.00      0.97      0.99       459
           4       0.97      1.00      0.99       384

    accuracy                           0.99      2070
   macro avg       0.99      0.99      0.99      2070
weighted avg       0.99      0.99      0.99      2070

